# Tool Approval with Human-in-the-Loop

This notebook demonstrates **human-in-the-loop tool approval** using the `agent_framework` library.

When a tool is decorated with `@tool(approval_mode="always_require")`, the agent pauses before
executing the tool and asks the user to approve or reject the call.

## Bug Fix Applied

**Root cause:** The `agent_framework` library serializes `function_call` history items with
`"status": null` when resuming after approvals. The Azure AI Foundry Responses API requires this
field to be a valid string (`"in_progress"`, `"completed"`, or `"incomplete"`), causing a
`400 BadRequestError: invalid_payload` on the second call.

**Fix:** A targeted monkey-patch intercepts `_prepare_content_for_openai` and sets
`"status": "completed"` for any `function_call` item that would otherwise have `null` status.

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from typing import TYPE_CHECKING, Annotated, Any
from agent_framework import tool, Message, AgentResponse
from agent_framework.azure import AzureOpenAIResponsesClient
from azure.identity.aio import AzureCliCredential
from random import randrange

if TYPE_CHECKING:
    from agent_framework import SupportsAgentRun

## Bug Fix: Patch `function_call` Status Serialization

The cell below patches the library at runtime. It must be run **once** before creating the agent.

In [ ]:
# Bug fix: agent_framework serializes function_call items with "status": None when building
# the conversation history for approval continuations. The Azure AI Foundry Responses API
# rejects this with a 400 ValidationError because status must be one of:
#   "in_progress" | "completed" | "incomplete"
#
# Patch: intercept _prepare_content_for_openai and set status = "completed" for function_calls
# that have a null status (i.e. calls already decided by the model in a previous turn).

from agent_framework.openai._responses_client import RawOpenAIResponsesClient

_orig_prepare_content = RawOpenAIResponsesClient._prepare_content_for_openai

def _patched_prepare_content(self, role, content, call_id_to_id):
    result = _orig_prepare_content(self, role, content, call_id_to_id)
    if isinstance(result, dict) and result.get("type") == "function_call" and result.get("status") is None:
        result["status"] = "completed"
    return result

RawOpenAIResponsesClient._prepare_content_for_openai = _patched_prepare_content
print("Patch applied.")

## Define Tools

Both tools require explicit user approval before execution (`approval_mode="always_require"`).

In [ ]:
@tool(approval_mode="always_require")
def get_weather(location: Annotated[str, "The city and state, e.g. San Francisco, CA"]) -> str:
    """Get the current weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [ ]:
@tool(approval_mode="always_require")
def get_weather_detail(location: Annotated[str, "The city and state, e.g. San Francisco, CA"]) -> str:
    """Get the current weather for a given location."""
    conditions = ["sunny", "cloudy", "raining", "snowing", "clear"]

    return (
        f"The weather in {location} is {conditions[randrange(0, len(conditions))]} and {randrange(-10, 30)}°C, "
        "with a humidity of 88%. "
        f"Tomorrow will be {conditions[randrange(0, len(conditions))]} with a high of {randrange(-10, 30)}°C."
    )

## Set Up Client and Agent

In [ ]:
load_dotenv(override=True)

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

In [ ]:
credential = AzureCliCredential()
client = AzureOpenAIResponsesClient(
    project_endpoint=project_endpoint,
    deployment_name=model,
    credential=credential,
)

agent = client.as_agent(
    name="WeatherAgent",
    instructions="""
        You are a helpful weather assistant. 
        Use the get_weather tool to provide weather information.
        Use the get_weather_detail tool to provide detailed weather information.
        """,
    tools=[get_weather, get_weather_detail],
)

## Human-in-the-Loop Approval Handler

The `handle_approvals` function drives the approval loop:

1. Run the agent — it pauses when a tool with `approval_mode="always_require"` is triggered.
2. For each pending tool call, print the function name and arguments, then prompt the user.
3. Rebuild the conversation with the user's approval/rejection responses and re-run the agent.
4. Repeat until no approval requests remain.

In [ ]:
async def handle_approvals(query: str, agent: "SupportsAgentRun") -> AgentResponse:
    result = await agent.run(query)
    
    while len(result.user_input_requests) > 0:
        # Start with the original query
        new_inputs: list[Any] = [query]

        for user_input_needed in result.user_input_requests:
            print(
                f"\nUser Input Request for function from {agent.name}:"
                f"\n  Function: {user_input_needed.function_call.name}"
                f"\n  Arguments: {user_input_needed.function_call.arguments}"
            )

            # Add the assistant message with the approval request
            new_inputs.append(Message("assistant", [user_input_needed]))

            # Get user approval
            user_approval = await asyncio.to_thread(input, "\nApprove function call? (y/n): ")

            # Add the user's approval response
            new_inputs.append(
                Message("user", [user_input_needed.to_function_approval_response(user_approval.lower() == "y")])
            )

        # Run again with all the context (patch ensures function_call status is valid)
        result = await agent.run(new_inputs)

    return result

## Run the Agent

In [ ]:
query = "Can you give me an update of the weather in LA and Portland and detailed weather for Seattle?"
print(f"User: {query}")

result = await handle_approvals(query, agent)

In [ ]:
print(result)